# 🛰️ SIH 2026 — PS 26143: Sentinel-1 Oil-Spill Dataset Inspection

> **Member 1 scope:** Satellite Imagery + AI Oil-Spill Detection  
> **Phase:** Dataset Inspection (pre-training)  
> **Model training:** ❌ Not performed in this notebook

---

This notebook:
1. Installs required libraries (Colab-safe)
2. Clones / mounts the project (Colab-safe)
3. Detects the dataset directory structure
4. Lists all image and mask files
5. Pairs images ↔ masks by filename
6. Reports image dimensions, channels, and dtypes
7. Displays sample images + masks + overlays
8. Reports missing / unmatched pairs
9. Prints a full dataset summary

---

### ⚠️ Before running
Place the Sentinel-1 dataset inside `ml/dataset/` (see `ml/dataset/README.md`).  
In Colab you can upload via the cell below or mount Google Drive.

## 0. Environment Setup

In [ ]:
# ── Install required libraries (Colab-safe) ──────────────────────────────────
import importlib, subprocess, sys

REQUIRED = {
    "rasterio": "rasterio",
    "numpy":    "numpy",
    "PIL":      "Pillow",
    "tqdm":     "tqdm",
    "matplotlib": "matplotlib",
}

for module, package in REQUIRED.items():
    if importlib.util.find_spec(module) is None:
        print(f"Installing {package} …")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    else:
        print(f"✓ {package} already available")

In [ ]:
# ── Detect runtime environment ────────────────────────────────────────────────
import os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
print(f"Running in Google Colab : {IN_COLAB}")

In [ ]:
# ── (Colab only) Mount Google Drive ──────────────────────────────────────────
# Skip this cell if you are running locally or if you have already mounted.

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted at /content/drive")
else:
    print("Not in Colab — skipping Drive mount.")

In [ ]:
# ── Locate the project root ───────────────────────────────────────────────────
# Colab  : clone the repo from GitHub (edit the URL below), OR
#          set REPO_PATH to point to a Drive-synced copy.
# Local  : the project root is detected automatically.

GITHUB_REPO_URL = ""  # e.g. "https://github.com/your-org/sih26143-oil-spill.git"
                       # Leave blank to skip cloning.

if IN_COLAB:
    if GITHUB_REPO_URL:
        import subprocess
        subprocess.run(["git", "clone", GITHUB_REPO_URL, "/content/sih26143-oil-spill"], check=True)
        PROJECT_ROOT = Path("/content/sih26143-oil-spill")
    else:
        # Assume repo is already available at /content/sih26143-oil-spill
        # (e.g. you uploaded it or synced via Drive)
        PROJECT_ROOT = Path("/content/sih26143-oil-spill")
        if not PROJECT_ROOT.exists():
            print("\n⚠️  Project root not found at /content/sih26143-oil-spill")
            print("Set GITHUB_REPO_URL above, or upload the project manually.")
else:
    # Local: walk up from this notebook to find the project root
    _nb_dir = Path().resolve()
    PROJECT_ROOT = next(
        (p for p in [_nb_dir, _nb_dir.parent, _nb_dir.parent.parent]
         if (p / 'ml').exists()),
        _nb_dir
    )

print(f"Project root : {PROJECT_ROOT}")

# Add ml/ to Python path so imports work
ML_DIR = PROJECT_ROOT / 'ml'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"ml/ directory: {ML_DIR}")
print(f"Exists       : {ML_DIR.exists()}")

## 1. Configure Dataset Path

**Set the `DATASET_ROOT` variable below to point to your dataset.**

| Scenario | Value |
|---|---|
| Dataset in `ml/dataset/` | *(default, no change needed)* |
| Dataset in Google Drive | `/content/drive/MyDrive/<your-folder>` |
| Dataset uploaded to Colab | `/content/<unzipped-folder>` |

In [ ]:
# ── Set dataset root ──────────────────────────────────────────────────────────
# Default: ml/dataset/ (relative to project root)
DATASET_ROOT = ML_DIR / "dataset"

# ── Override examples (uncomment and edit as needed) ──────────────────────────
# DATASET_ROOT = Path("/content/drive/MyDrive/SIH2026/sentinel1_oil_spill")
# DATASET_ROOT = Path("/content/sentinel1_oil_spill")

print(f"Dataset root : {DATASET_ROOT}")
print(f"Exists       : {DATASET_ROOT.exists()}")

if not DATASET_ROOT.exists():
    print("\n" + "═"*60)
    print("  ⚠  DATASET NOT FOUND")
    print("═"*60)
    print("Please either:")
    print("  1. Place the dataset in  ml/dataset/  (see ml/dataset/README.md)")
    print("  2. Set DATASET_ROOT above to the correct path")
    print("")
    print("Continuing will raise FileNotFoundError.")

### (Optional) Upload dataset zip directly in Colab

In [ ]:
# ── (Colab only) Upload and unzip dataset ────────────────────────────────────
# Run this cell ONLY if you want to upload a zip file directly.
# Leave UPLOAD = False to skip.

UPLOAD = False  # ← Set to True to trigger file upload dialog

if UPLOAD and IN_COLAB:
    from google.colab import files
    import zipfile

    print("Select your dataset zip file …")
    uploaded = files.upload()

    for fname, data in uploaded.items():
        zip_path = Path("/content") / fname
        zip_path.write_bytes(data)
        print(f"Uploaded: {zip_path}")

        if zipfile.is_zipfile(zip_path):
            extract_to = Path("/content/dataset_extracted")
            extract_to.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall(extract_to)
            print(f"Extracted to: {extract_to}")
            DATASET_ROOT = extract_to
        else:
            print("Uploaded file is not a zip — set DATASET_ROOT manually.")
elif UPLOAD and not IN_COLAB:
    print("Upload is only supported in Google Colab.")
else:
    print("Upload skipped (UPLOAD=False).")

## 2. Initialise the Dataset Inspector

In [ ]:
# ── Import the modular inspector ──────────────────────────────────────────────
from ml.preprocessing.dataset_inspector import DatasetInspector

inspector = DatasetInspector(DATASET_ROOT, verbose=True)
print("DatasetInspector initialised.")

## 3. Detect Directory Structure

In [ ]:
structure = inspector.detect_structure()

## 4. List and Classify All Files

In [ ]:
file_groups = inspector.list_files()

print(f"\nFirst 10 IMAGE files:")
for p in file_groups['images'][:10]:
    print(f"  {p.relative_to(DATASET_ROOT)}")

print(f"\nFirst 10 MASK files:")
for p in file_groups['masks'][:10]:
    print(f"  {p.relative_to(DATASET_ROOT)}")

## 5. Pair Images ↔ Masks

In [ ]:
pairs = inspector.pair_images_and_masks()

print(f"\nFirst 5 pairs:")
for img, mask in pairs[:5]:
    img_rel  = img.relative_to(DATASET_ROOT)
    mask_rel = mask.relative_to(DATASET_ROOT) if mask else None
    print(f"  IMG : {img_rel}")
    print(f"  MSK : {mask_rel}")
    print()

## 6. Inspect a Single File (dimensions, channels, dtype)

In [ ]:
# ── Inspect the first image and mask individually ─────────────────────────────
if pairs:
    first_img, first_mask = pairs[0]

    img_meta = inspector.inspect_file(first_img)
    print("Image metadata:")
    for k, v in img_meta.items():
        print(f"  {k:20s}: {v}")

    if first_mask:
        print()
        mask_meta = inspector.inspect_file(first_mask)
        print("Mask metadata:")
        for k, v in mask_meta.items():
            print(f"  {k:20s}: {v}")
else:
    print("No pairs found — check DATASET_ROOT.")

## 7. Display Sample Images + Masks + Overlays

For each sample the notebook shows 4 panels:

| Panel | Description |
|---|---|
| Image (ch0) | First channel, grayscale, 2–98 percentile normalised |
| Image (viridis / RGB) | Colourised — shows channel 0 or first 3 channels as RGB |
| Mask | Raw mask with `tab10` colour palette + unique values |
| Overlay | Mask overlaid on top of the image |

In [ ]:
# ── Display up to 4 samples ───────────────────────────────────────────────────
inspector.inspect_samples(n=4)

## 8. Manual File Load & Inspection (Advanced)

Use this cell to inspect any specific file in detail.

In [ ]:
import numpy as np
from ml.preprocessing.dataset_inspector import _load_array, _array_info, _normalise_for_display
import matplotlib.pyplot as plt

# ── Set the path to any raster file you want to inspect ──────────────────────
INSPECT_FILE = None   # e.g. DATASET_ROOT / "images" / "0001.tif"

if INSPECT_FILE is None and pairs:
    INSPECT_FILE = pairs[0][0]   # default: first image

if INSPECT_FILE:
    arr = _load_array(INSPECT_FILE)
    if arr is None:
        print(f"Could not load: {INSPECT_FILE}")
    else:
        info = _array_info(arr)
        print(f"File     : {INSPECT_FILE}")
        print(f"Shape    : {info['shape']}")
        print(f"Dtype    : {info['dtype']}")
        print(f"Channels : {info['channels']}")
        print(f"Min      : {info['min']:.6g}")
        print(f"Max      : {info['max']:.6g}")
        print(f"Mean     : {np.nanmean(arr):.6g}")
        print(f"Std      : {np.nanstd(arr):.6g}")

        # Show histogram
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

        # Image
        disp = _normalise_for_display(arr[..., 0] if arr.ndim == 3 else arr)
        ax1.imshow(disp, cmap='gray')
        ax1.set_title(f"Ch-0  |  shape={info['shape']}  |  dtype={info['dtype']}")
        ax1.axis('off')

        # Histogram
        flat = arr.flatten()
        ax2.hist(flat[~np.isnan(flat)], bins=128, color='steelblue', edgecolor='none')
        ax2.set_xlabel('Pixel value')
        ax2.set_ylabel('Count')
        ax2.set_title('Pixel Value Distribution (all channels)')
        ax2.grid(alpha=0.3)

        plt.tight_layout()
        plt.show()
else:
    print("No file to inspect. Set INSPECT_FILE or ensure DATASET_ROOT is correct.")

## 9. Mask Unique Values & Class Distribution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ml.preprocessing.dataset_inspector import _load_array
from collections import Counter

MAX_MASKS_TO_SCAN = 50   # adjust as needed

all_unique_vals = Counter()

mask_files = inspector.mask_files[:MAX_MASKS_TO_SCAN]
if not mask_files:
    # If no masks classified, try from pairs
    mask_files = [m for _, m in inspector.pairs if m is not None][:MAX_MASKS_TO_SCAN]

if not mask_files:
    print("No mask files found. Ensure dataset is correctly placed and classified.")
else:
    print(f"Scanning {len(mask_files)} mask files for unique values …")
    for mp in mask_files:
        arr = _load_array(mp)
        if arr is not None:
            for v in np.unique(arr).tolist():
                all_unique_vals[v] += 1

    print(f"\nUnique mask values across all scanned masks:")
    print(f"  Values  : {sorted(all_unique_vals.keys())}")
    print(f"  (value → number of masks it appears in)")
    for val, count in sorted(all_unique_vals.items()):
        print(f"    {val:8} → {count} mask(s)")

    # Bar chart
    fig, ax = plt.subplots(figsize=(8, 4))
    vals  = [str(v) for v in sorted(all_unique_vals.keys())]
    counts = [all_unique_vals[v] for v in sorted(all_unique_vals.keys())]
    ax.bar(vals, counts, color='steelblue')
    ax.set_xlabel('Mask class value')
    ax.set_ylabel(f'Number of masks (out of {len(mask_files)} scanned)')
    ax.set_title('Mask Class Value Distribution')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

## 10. Report Missing / Unmatched Pairs

In [ ]:
inspector.report_missing_pairs()

## 11. Full Dataset Summary

In [ ]:
inspector.print_summary()

In [ ]:
# ── Save summary to JSON ──────────────────────────────────────────────────────
summary_out = ML_DIR / "dataset_summary.json"
inspector.save_summary_json(summary_out)
print(f"Summary saved to: {summary_out}")

---

## ✅ Inspection Complete

Review the output above. Key things to note before moving to preprocessing:

| Question | Where to find the answer |
|---|---|
| How many SAR channels? | Section 6 — `channels` field |
| Image dtype (float32? uint16?) | Section 6 — `dtype` field |
| Are masks binary or multi-class? | Section 9 — unique mask values |
| How many matched image–mask pairs? | Section 11 — Summary |
| Any missing pairs? | Section 10 — Missing report |

**Next step:** `ml/preprocessing/` — normalisation, patching, augmentation scripts  
*(will be built in the next phase)*

---
*SIH 2026 — Problem Statement 26143 | Member 1: Satellite Imagery + AI Detection*